Importing libraries and setting print settings for outputs

In [1]:
import pandas as pd # importujemy pandas jako pd

pd.set_option('display.width', 1000, 'display.max_columns', None, 'display.expand_frame_repr', False)


Load .csv files and replace wrong filled data and errors

In [2]:
path = 'fact_subscriptions.csv'
path2 = 'dim_customers.csv'
table = pd.read_csv(path)
dim_customers = pd.read_csv(path2)

characters_to_replace=[
    [
        (" ", ""),
        ("act", "active"),
        ("ACT", "active"),
        ("active_plan", "active"),
        ("churned", "cancelled"),
        ("canceled", "cancelled")
    ],
    [
        ("United States", "US"),
        ("united kingdom", "UK"),
        ("Canada", "CA"),
        ("usa", "US")
    ],
    [
        ("Enterprise", "ENT"),
        ("small business", "SMB"),
        ("Mid-Market", "MID"), 
        ("mid", "MID")
    ],
    [
        ("P_PRO", "p_pro"),
        ("P_PRO ", "p_pro")
    ]
]
table['status'] = table['status'].replace(dict(characters_to_replace[0]))
table['plan_id'] = table['plan_id'].replace(dict(characters_to_replace[3]))

# print(table.plan_id.unique())
# print(table.status.unique())
# print(table.head())

dim_customers['country'] = dim_customers['country'].replace(dict(characters_to_replace[1]))
dim_customers['segment'] = dim_customers['segment'].replace(dict(characters_to_replace[2]))

all_dim_custmers = dim_customers.copy()

Assing data types to dataframes

In [3]:
table = table.astype({
    'subscription_id': 'string',         
    'customer_id': 'string',    
    'plan_id': 'category', 
    'start_date': 'datetime64[ns]',
    'end_date': 'datetime64[ns]',
    'status': 'category'
})

dim_customers = dim_customers.astype({
    'customer_id': 'string',  
    'signup_date': 'datetime64[ns]', 
    'country': 'category',
    'industry': 'category',
    'company_size': 'category',
    'segment': 'category'
})

all_dim_custmers = all_dim_custmers.astype({
    'customer_id': 'string',  
    'signup_date': 'datetime64[ns]', 
    'country': 'category',
    'industry': 'category',
    'company_size': 'category',
    'segment': 'category'
})

# print(table)

Preparing data to analysis 

In [4]:
# Filter to identify active customers and remove invalid data where the start_date is greater than the end_date
table = table[table['end_date'].isna() | (table['end_date'] >= table['start_date'])]

# Sorting table
table = table.sort_values(by = 'start_date', ascending = True) 

# Filtering data by the required financial year
segments_2025 = table[table['start_date'].between('2025-01-01', '2025-12-31')]
churned_2025 = segments_2025[segments_2025['end_date'] <= '2025-12-31']
actual_customers = table[table['end_date'].isna()]

Take a look at data after filtering

In [5]:
print('\nTable')
table.info()

print('\nSegments_2025 table')
segments_2025.info()

print('\nChurned_2025 table')
churned_2025.info()


Table
<class 'pandas.core.frame.DataFrame'>
Index: 1477 entries, 919 to 411
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   subscription_id  1477 non-null   string        
 1   customer_id      1477 non-null   string        
 2   plan_id          1477 non-null   category      
 3   start_date       1477 non-null   datetime64[ns]
 4   end_date         434 non-null    datetime64[ns]
 5   status           1477 non-null   category      
dtypes: category(2), datetime64[ns](2), string(2)
memory usage: 60.8 KB

Segments_2025 table
<class 'pandas.core.frame.DataFrame'>
Index: 1475 entries, 919 to 733
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   subscription_id  1475 non-null   string        
 1   customer_id      1475 non-null   string        
 2   plan_id          1475 non-null   category      
 3   st

In [6]:
table.describe(include='all')

,subscription_id,customer_id,plan_id,start_date,end_date,status
count,1477,1477,1477,1477,434,1477
unique,1477,1477,3,NaN,NaN,2
top,SUB_30919,CUST_2919,p_pro,NaN,NaN,active
freq,1,1,738,NaN,NaN,1043
mean,NaN,NaN,NaN,2025-07-05 01:33:35.707514880,2025-11-10 23:06:54.746543872,NaN
min,NaN,NaN,NaN,2025-01-01 00:00:00,2025-02-10 00:00:00,NaN
25%,NaN,NaN,NaN,2025-04-05 00:00:00,2025-08-17 00:00:00,NaN
50%,NaN,NaN,NaN,2025-07-11 00:00:00,2025-11-11 00:00:00,NaN
75%,NaN,NaN,NaN,2025-10-03 00:00:00,2026-02-07 18:00:00,NaN
max,NaN,NaN,NaN,2026-01-01 00:00:00,2026-07-30 00:00:00,NaN


In [7]:
segments_2025.describe(include='all')

,subscription_id,customer_id,plan_id,start_date,end_date,status
count,1475,1475,1475,1475,434,1475
unique,1475,1475,3,NaN,NaN,2
top,SUB_30919,CUST_2919,p_pro,NaN,NaN,active
freq,1,1,737,NaN,NaN,1041
mean,NaN,NaN,NaN,2025-07-04 19:42:15.864406784,2025-11-10 23:06:54.746543872,NaN
min,NaN,NaN,NaN,2025-01-01 00:00:00,2025-02-10 00:00:00,NaN
25%,NaN,NaN,NaN,2025-04-05 00:00:00,2025-08-17 00:00:00,NaN
50%,NaN,NaN,NaN,2025-07-11 00:00:00,2025-11-11 00:00:00,NaN
75%,NaN,NaN,NaN,2025-10-03 00:00:00,2026-02-07 18:00:00,NaN
max,NaN,NaN,NaN,2025-12-31 00:00:00,2026-07-30 00:00:00,NaN


In [8]:
churned_2025.describe(include='all')

,subscription_id,customer_id,plan_id,start_date,end_date,status
count,277,277,277,277,277,277
unique,277,277,3,NaN,NaN,1
top,SUB_30919,CUST_2919,p_pro,NaN,NaN,cancelled
freq,1,1,136,NaN,NaN,277
mean,NaN,NaN,NaN,2025-05-03 10:13:25.776173312,2025-08-30 16:58:55.018050560,NaN
min,NaN,NaN,NaN,2025-01-01 00:00:00,2025-02-10 00:00:00,NaN
25%,NaN,NaN,NaN,2025-02-27 00:00:00,2025-07-03 00:00:00,NaN
50%,NaN,NaN,NaN,2025-04-26 00:00:00,2025-09-07 00:00:00,NaN
75%,NaN,NaN,NaN,2025-06-25 00:00:00,2025-10-30 00:00:00,NaN
max,NaN,NaN,NaN,2025-11-28 00:00:00,2025-12-30 00:00:00,NaN


# Churn, CLV and ARPU

Churn for all

In [9]:
#print(segments_2025[segments_2025['status'] == 'active'])

# Check data counts
print(segments_2025.count(), segments_2025.start_date.count(), segments_2025.end_date.count(), churned_2025.end_date.count(), segments_2025.end_date.count() - churned_2025.end_date.count(), sep="\n")

# Churn rate for all
churn_rate_2025 = churned_2025.end_date.count() / segments_2025.start_date.count()
print(f'Annual churn rate for all is {churn_rate_2025.round(2) * 100}% in 2025')


subscription_id    1475
customer_id        1475
plan_id            1475
start_date         1475
end_date            434
status             1475
dtype: int64
1475
434
277
157
Annual churn rate for all is 19.0% in 2025


Churn,retention and power (share) for segments

In [10]:
# Churn by Segments
segments_churn_by_plan = table.groupby('plan_id')['status'].value_counts(normalize=True) * 100
print('\n', segments_churn_by_plan.round(2))

# Shares of tier plans (Not revenue just count)
segments_agg = table.groupby('plan_id')['subscription_id'].agg(count='count').reset_index()
# We are sorting the lines according to the new numbering order
segments_agg['plans_shares_in_%'] = ((segments_agg['count'] / segments_agg['count'].sum()) * 100).round(2) 

# Enter the line numbers in brackets in the order in which they are to appear:
segments_agg = segments_agg.iloc[[0, 2, 1]].reset_index(drop=True)
print('\n', segments_agg)



 plan_id  status   
p_basic  active       71.71
         cancelled    28.29
p_ent    active       71.21
         cancelled    28.79
p_pro    active       69.78
         cancelled    30.22
Name: proportion, dtype: float64

    plan_id  count  plans_shares_in_%
0  p_basic    350              23.70
1    p_pro    738              49.97
2    p_ent    389              26.34


C:\Users\Rafael\AppData\Local\Temp\ipykernel_5516\3573384491.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  segments_churn_by_plan = table.groupby('plan_id')['status'].value_counts(normalize=True) * 100
C:\Users\Rafael\AppData\Local\Temp\ipykernel_5516\3573384491.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  segments_agg = table.groupby('plan_id')['subscription_id'].agg(count='count').reset_index()


Filter data for Customer Life Time for all plan's segments and count it and describe it

In [11]:
# Customer lifetime for all segments in 2025
all_churned = table[table['status'] == 'cancelled']
all_churned['lifetime'] = all_churned['end_date'] - all_churned['start_date'] 
print('Customer Life Time for all segments in 2025\n', all_churned['lifetime'].dt.days.describe(include='all'))


Customer Life Time for all segments in 2025
 count    434.000000
mean     133.313364
std       62.637256
min       30.000000
25%       77.000000
50%      133.000000
75%      188.750000
max      240.000000
Name: lifetime, dtype: float64


C:\Users\Rafael\AppData\Local\Temp\ipykernel_5516\3575128151.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  all_churned['lifetime'] = all_churned['end_date'] - all_churned['start_date']


Customer Life Time Value for each plan's segments and Avarage Life Time Value for all cohorts

In [12]:
# Customer life time for segments in 2025

# Grouping by segments (subscritpion tiers), then count mean from days then convert it to number
avg_lifetime_by_plan = all_churned.groupby('plan_id')['lifetime'].mean().dt.total_seconds() / 86400

# Reorder and count a mointh from days assuming month has 30 days
avg_lifetime_by_plan = avg_lifetime_by_plan.reindex(['p_basic', 'p_pro', 'p_ent']) / 30

# Converts to a DataFrame and names the column containing the values
avg_lifetime_by_plan = avg_lifetime_by_plan.reset_index(name='Customer LifeTime in months')
print('\nAVG_LIFETIME_BY_PLAN\n', avg_lifetime_by_plan)

# Customer life time value per segments in dolars, customer live time for specific tier multiplied by tiers
CLV = (avg_lifetime_by_plan['Customer LifeTime in months'] * [29.0, 79.0, 499.0 / 12]).to_frame(name='Customer Monthly Life Time Value') #Here is the catch if we have policy to return rest of the money for annual payment for P_ent plans we have to count it monthly, that is  why i devideed it by 12
# Zmienia 'stara_nazwa' na 'nowa_nazwa'
print('\nIt is a CLV\n', CLV, '\n', type(CLV),)

# Putting it all together and reset indexes to add them and move plan_id to the column names
customer_life_time_details_table = pd.DataFrame({
    'plan_id': avg_lifetime_by_plan['plan_id'],
    'Montly_lifetime_by_plan': avg_lifetime_by_plan['Customer LifeTime in months'],
    'Price_for_tier_monthly': [29.0, 79.0, 499.0 / 12], # Here is the catch if we have policy to return rest of the money for annual payment for P_ent plans we have to count it monthly, that is  why i devideed it by 12
    'Customer_Lifetime_Value': CLV['Customer Monthly Life Time Value']
}).round(2)
print('DETAILED TABLE\n',customer_life_time_details_table)



AVG_LIFETIME_BY_PLAN
    plan_id  Customer LifeTime in months
0  p_basic                     4.895623
1    p_pro                     4.291330
2    p_ent                     4.347917

It is a CLV
    Customer Monthly Life Time Value
0                        141.973064
1                        339.015097
2                        180.800868 
 <class 'pandas.core.frame.DataFrame'>
DETAILED TABLE
    plan_id  Montly_lifetime_by_plan  Price_for_tier_monthly  Customer_Lifetime_Value
0  p_basic                     4.90                   29.00                   141.97
1    p_pro                     4.29                   79.00                   339.02
2    p_ent                     4.35                   41.58                   180.80


C:\Users\Rafael\AppData\Local\Temp\ipykernel_5516\1462502447.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  avg_lifetime_by_plan = all_churned.groupby('plan_id')['lifetime'].mean().dt.total_seconds() / 86400


Preparing DataFrames, cohort churn and life time

In [13]:
cohort_analysis = pd.DataFrame()
cohort_analysis['customer_id'] = table['customer_id']
cohort_analysis['cohort_month'] = table['start_date'].dt.to_period('M')
cohort_analysis['churned_date'] = table['end_date'].dt.to_period('M')

# Sorting and counting the `cohort_month` values so that we can then calculate the number of departures from that figure
cohort_month_base = cohort_analysis['cohort_month'].value_counts().reset_index().sort_values(by = 'cohort_month', ascending = True)
cohort_month_base = cohort_month_base.rename(columns={'count': 'customers_count'})

# cohort_month_base['1_month_of_using'] = cohort_month_base['count'] - "this group churner month +1" / cohort_month_base['count']

cohort_churn_after_months = pd.DataFrame()
# Calculating the churn rate for each cohort
for i in cohort_month_base['cohort_month']:
    # print(i)
    if i == cohort_analysis['churned_date'].max():
        continue;
    for j in range(cohort_month_base['cohort_month'].nunique() - 1):
        if j == 0:
            continue;
        cohort_churn_after_months = pd.concat([cohort_churn_after_months, cohort_analysis[(cohort_analysis['cohort_month'] == i) & (cohort_analysis['churned_date'] == i+j)][['cohort_month', 'churned_date']].value_counts().reset_index().sort_values(by='cohort_month', ascending=True)], ignore_index=True)
cohort_churn_after_months = cohort_churn_after_months.rename(columns={'count': 'churned_count'})

# We iterate through all the months, using `.unique()` to check for duplicates 
# in the main database (if the values do not repeat, `.unique()` is simply an optional safeguard)
for i in cohort_month_base['cohort_month'].unique():
    # Condition: Is month “i” missing from the cohort_churn_after_months DataFrame?
    if i not in cohort_churn_after_months['cohort_month'].values:
        # If it’s missing, we’ll add it to the table as we discussed
        cohort_churn_after_months.loc[len(cohort_churn_after_months)] = {
            'cohort_month': i,
            'churned_date': i + 1,
            'churned_count': 0
        }
# Calculating how long a cohort of customers used the product before leaving - Customer Life Time
cohort_churn_after_months['month_of_using'] = cohort_churn_after_months['churned_date'].astype('int64') - cohort_churn_after_months['cohort_month'].astype('int64')

Getting sum of customers after churn and customer stayed in procent

In [14]:
c = 0
sum_of_customers_after_churn = []
customer_stayed_in_procent = []

for k, h in zip(cohort_month_base['cohort_month'], cohort_month_base['customers_count']):
    c = 0
    for i, j in zip(cohort_churn_after_months['cohort_month'], cohort_churn_after_months['churned_count']):
        if i == k: 
            c += j
            sum_of_customers_after_churn.append(h - c)
            customer_stayed_in_procent.append((h - c) / h)
            # print(i, k, j, c)
# print(sum_churn)
cohort_churn_after_months['sum_of_customers_after_churn'] = sum_of_customers_after_churn
cohort_churn_after_months['customer_stayed_in_procent'] = customer_stayed_in_procent
# print(cohort_churn_after_months.head(10))
# print(cohort_churn_after_months.tail(10))    

Save dataframe to excel file to use pivot tables on churn data

In [15]:
with pd.ExcelWriter('cochort_analysis.xlsx') as writer:
    cohort_churn_after_months.to_excel(writer, sheet_name='Churn_by_month_of_using', index=False)
    cohort_month_base.to_excel(writer, sheet_name='Number_of_customers_in_cochorts', index=False)

Finding best and rest cohors to compare them

In [16]:
best_performed_cohorts = cohort_analysis[cohort_analysis['cohort_month'].between('2025-07', '2025-11')]
dim_customers = dim_customers[dim_customers['customer_id'].isin(best_performed_cohorts['customer_id'])]
# print(best_performed_cohorts)
# print(dim_customers)

Summary of best cohorts -  Take a look at the best cohorts with smallest churn toghether

In [17]:
print((dim_customers.segment.value_counts(normalize=True) * 100).round(2), '\n')
print((dim_customers.industry.value_counts(normalize=True) * 100).round(2), '\n')
print((dim_customers.company_size.value_counts(normalize=True) * 100).round(2), '\n')
print((dim_customers.country.value_counts(normalize=True) * 100).round(2), '\n')


segment
MID    38.36
SMB    30.98
ENT    30.66
Name: proportion, dtype: float64 

industry
Healthcare    22.63
SaaS          22.44
E-commerce    19.73
Finance       18.57
Logistics     16.63
Name: proportion, dtype: float64 

company_size
51-200     20.22
1-10       17.66
11-50      16.85
500+       15.89
unknown    15.41
201-500    13.96
Name: proportion, dtype: float64 

country
US    46.88
CA    27.71
UK    25.42
Name: proportion, dtype: float64 



Comparing segments in cohorts

In [18]:
# ==============================================================================
# 1. CONFIGURATION AND TIME PROCESSING (WITH CORRECT MERGE)
# ==============================================================================
cohorts_list = sorted(best_performed_cohorts['cohort_month'].unique().tolist())
# Time formatting
dim_customers['signup_date'] = pd.to_datetime(dim_customers['signup_date'])
dim_customers['cohort_month'] = dim_customers['signup_date'].dt.to_period('M')
# Merging and extracting dates from the MAIN AND UNCUT DATABASE (all_dim → all_dim)
all_dim_custmers = all_dim_custmers.merge(table, on="customer_id", how='left')
all_dim_custmers['signup_date'] = pd.to_datetime(all_dim_custmers['signup_date'])
all_dim_custmers['cohort_month'] = all_dim_custmers['signup_date'].dt.to_period('M')

# ==============================================================================
# 2. DEMOGRAPHIC PROFILING AND DISTRIBUTION OF PLANS
# ==============================================================================
seg = (all_dim_custmers.groupby('cohort_month')['segment'].value_counts(normalize=True).unstack(0) * 100).round(2)
ind = (all_dim_custmers.groupby('cohort_month')['industry'].value_counts(normalize=True).unstack(0) * 100).round(2)
size = (all_dim_custmers.groupby('cohort_month')['company_size'].value_counts(normalize=True).unstack(0) * 100).round(2)
coun = (all_dim_custmers.groupby('cohort_month')['country'].value_counts(normalize=True).unstack(0) * 100).round(2)
tabela_bazowa = pd.concat([seg, ind, size, coun], keys=['Segment', 'Industry', 'Company Size', 'Country']).fillna(0)

inne_kohorty = [miesiac for miesiac in tabela_bazowa.columns if miesiac not in cohorts_list]

plan_all = (all_dim_custmers.groupby('cohort_month')['plan_id'].value_counts(normalize=True).unstack(0) * 100).round(2)
plan_top = plan_all[cohorts_list].copy()
plan_top['Avg. TOP 5'] = plan_top.mean(axis=1).round(2)
plan_rest = plan_all[[m for m in plan_all.columns if m not in cohorts_list]].copy()
plan_rest['Avg. rest'] = plan_rest.mean(axis=1).round(2)

# ==============================================================================
# 3. ANALYTICAL SUMMARY OF CONTRAST TABLES (TOP 5 vs CONTROL GROUP)
# ================================================================================
tabela_top = tabela_bazowa[cohorts_list].copy()
tabela_top['Avg. from TOP 5 cohorts'] = tabela_top[cohorts_list].mean(axis=1).round(2)
tabela_top['Standard deviation'] = tabela_top[cohorts_list].std(axis=1).round(2)
tabela_pozostale = tabela_bazowa[inne_kohorty].copy()
tabela_pozostale['Avg. from rest'] = tabela_pozostale[inne_kohorty].mean(axis=1).round(2)
tabela_pozostale['Standard deviation'] = tabela_pozostale[inne_kohorty].std(axis=1).round(2)
tabela_porownanie = pd.DataFrame({
    'Avg. from TOP 5 cohorts': tabela_top['Avg. from TOP 5 cohorts'],
    'Avg. from rest': tabela_pozostale['Avg. from rest']
}).fillna(0)
tabela_porownanie['Difference (Top 5 – Others)'] = (tabela_porownanie['Avg. from TOP 5 cohorts'] - tabela_porownanie['Avg. from rest']).round(2)


# ==============================================================================
# 4. DISPLAYING A BREAKDOWN OF CUSTOMERS
# ==============================================================================
print("[TABLE 1] OVERVIEW OF THE BEST COHORTS (Your TOP 5):")
print(tabela_top)
print("[TABLE 2] CONTROL GROUP PROFILE (This time, it will also show the missing months):")
print(tabela_pozostale)
print("[TABLE 3] FINAL COMPARISON (What makes success stand out?):")
print(tabela_porownanie)
print("TOP SCHEDULE:")
print(plan_top)
print("SCHEDULE OF OTHER PLANS:")
print(plan_rest)

[TABLE 1] OVERVIEW OF THE BEST COHORTS (Your TOP 5):
cohort_month             2025-07  2025-08  2025-09  2025-10  2025-11  Avg. from TOP 5 cohorts  Standard deviation
Segment      ENT           27.56    32.75    26.92    32.12    32.80                    30.43                2.93
             MID           43.31    32.75    46.15    35.04    34.40                    38.33                5.99
             SMB           29.13    34.50    26.92    32.85    32.80                    31.24                3.11
Industry     E-commerce    19.05    23.08    25.45    15.79    15.69                    19.81                4.37
             Finance       25.71    19.58    17.27    21.05    15.69                    19.86                3.87
             Healthcare    17.14    17.48    26.36    25.44    25.49                    22.38                4.65
             Logistics     13.33    18.18    11.82    19.30    18.63                    16.25                3.42
             SaaS          24.76   

Counting Average Revenue Per User and Life Time Value for 3 the best and the worst cohorts, cause from coparision we did not have much difference.

In [19]:
# ==============================================================================
# PRICE POINTS AND CALCULATION OF INDIVIDUAL ARPU FOR COHORTS AND LTV FOR THE 3 BEST AND WORST COHORTS
# ==============================================================================
pricing = {'p_basic': 29.0, 'p_pro': 79.0, 'p_ent': 499.0}
# We store the price list as floating-point numbers for safety and calculate the average
all_dim_custmers['pricing'] = all_dim_custmers['plan_id'].map(pricing).astype(float)
monthly_arpu = all_dim_custmers.groupby('cohort_month')['pricing'].mean().round(2)

# ==============================================================================
# LTV SECTION – ESTIMATED REVENUE PER CUSTOMER (Based on hard metrics)
# ==============================================================================
ltv_table = pd.DataFrame({'ARPU': monthly_arpu})
retention_rate = {
    '2025-01': 0.672131147540984,
    '2025-03': 0.672413793103448, 
    '2025-04': 0.694915254237288, 
    '2025-05': 0.663716814159292,
    '2025-06': 0.683760683760684, 
    '2025-07': 0.760330578512397, 
    '2025-08': 0.725, 
    '2025-09': 0.739837398373984, 
    '2025-10': 0.736842105263158,
    '2025-11': 0.711864406779661
}
# KEY CHANGE: We’re converting the text index to the ‘Period(M)’ type so that Pandas can map it!
retencja_series = pd.Series(retention_rate)
retencja_series.index = pd.PeriodIndex(retencja_series.index, freq='M')

# LTV, churn and retention
ltv_table['Retention'] = retencja_series
ltv_table = ltv_table.dropna()
ltv_table['Churn_Rate'] = 1 - ltv_table['Retention']
ltv_table['LTV ($)'] = (ltv_table['ARPU'] / ltv_table['Churn_Rate']).round(2)
ltv_table = ltv_table.sort_values(by='LTV ($)', ascending=False)
avg_ltv_difference = ltv_table['LTV ($)'].head(5).mean() - ltv_table['LTV ($)'].tail(5).mean()

# print("\nDISTRIBUTION OF PLANS FOR TOP COHORTS:")
print(plan_top)

# print("\nDISTRIBUTION OF PLANS FOR REST:")
print(plan_rest)

# print("\nTHE DIFFERENCE BETWEEN THE TOP 5 AND THE REST COHORTS OF PLANS:")
print((plan_top['Avg. TOP 5'] - plan_rest['Avg. rest']).sort_values(ascending=False))

# print("\nAverage Revenue per User (ARPU) per month:")
print(monthly_arpu.to_frame('arpu').sort_values(by='arpu', ascending=False))

# print("\n[BUSINESS RESULTS] - Estimated cash inflow per user (LTV):")
print(ltv_table[['ARPU', 'Retention', 'Churn_Rate', 'LTV ($)']])

print(f"\nAvg Life Time Value difference (Avg. from TOP5 - Avg. from BOTTOM 5)\n{avg_ltv_difference}")

cohort_month  2025-07  2025-08  2025-09  2025-10  2025-11  Avg. TOP 5
plan_id                                                              
p_basic         19.64    24.16    20.00    29.69    19.83       22.66
p_ent           24.11    22.82    29.57    27.34    28.45       26.46
p_pro           56.25    53.02    50.43    42.97    51.72       50.88
cohort_month  2025-01  2025-02  2025-03  2025-04  2025-05  2025-06  2025-12  Avg. rest
plan_id                                                                               
p_basic         30.43    23.85    20.00    27.27    26.47    20.69    26.05      24.97
p_ent           28.70    21.10    29.57    23.64    24.51    26.72    26.89      25.88
p_pro           40.87    55.05    50.43    49.09    49.02    52.59    47.06      49.16
plan_id
p_pro      1.72
p_ent      0.58
p_basic   -2.31
dtype: float64
                arpu
cohort_month        
2025-03       193.17
2025-09       193.17
2025-11       188.57
2025-01       184.30
2025-06       180.

Get difference in procenteg for segments from cohorts comparision.

In [20]:
print(plan_top['Avg. TOP 5'])
print(plan_rest['Avg. rest'])
print(38.33 / 33.51 - 1)
print(22.38 / 18.51 - 1)
print(19.38 / 15.07 - 1)
print(47.61 / 42.93 - 1)

plan_id
p_basic    22.66
p_ent      26.46
p_pro      50.88
Name: Avg. TOP 5, dtype: float64
plan_id
p_basic    24.97
p_ent      25.88
p_pro      49.16
Name: Avg. rest, dtype: float64
0.14383766039988055
0.20907617504051856
0.2859986728599866
0.1090146750524108


# Recommendation
### I am recommending to target MID, 51-200 size, healthcare companies in US market basing on data proofs:
1. Best cohorts' LTV is higher on average 123$
2. Pro plan has on average 1.72% more customers and basic plan has 2.31 less customers in the best cohorts
3. Shares are higher for MID segment 14%, for 51-200 size 29%, for healthcare 21% and for US 11% on average compering best cohorts to poor cohorts

### What risks we may face:
1. Difference in cohorts segments are quite small few percentage points, may be not statistically significant cause we have lack of data.

### Next steps are:
1. Prepare survey to make sure we will be targeting correct one
2. Start preparing new marketing strategy targeting MID Healthcare companies (51-200 employees) in the US market
3. Start to develop converting incentives for basic plan conversion to pro plan
4. Monitor retention rate monthly to validate if targeting changes improve cohort performance

# Data Sources
https://www.kaggle.com/datasets/mounikabusi/saas-customer-churn-dataset
